In [1]:
import jax
import jax.numpy as jnp
from flax import nnx


class Linear(nnx.Module):
    def __init__(self, din: int, dout: int, *, rngs: nnx.Rngs):
        key = rngs.params()
        self.w = nnx.Param(jax.random.uniform(key, (din, dout)))
        self.b = nnx.Param(jnp.zeros((dout,)))
        self.din, self.dout = din, dout
        self.dropout = nnx.Dropout(0.5, rngs=rngs)

    def __call__(self, x: jax.Array):
        y = x @ self.w + self.b
        return self.dropout(y)


class Linear2(nnx.Module):
    def __init__(self, din: int, dout: int, *, rngs: nnx.Rngs):
        key = rngs.params()
        self.w = nnx.Param(jax.random.uniform(key, (din, dout)))
        self.b = nnx.Param(jnp.zeros((dout,)))
        self.din, self.dout = din, dout
        self.dropout = nnx.Dropout(0.5, rngs=rngs)
        self.lin = Linear(din, dout, rngs=rngs)

    def __call__(self, x: jax.Array):
        y = x @ self.w + self.b
        return self.dropout(y)


model = Linear2(2, 5, rngs=nnx.Rngs(2, dropout=42))
y = model(x=jnp.ones((1, 2)))

print(y)
nnx.display(model)

graphdef, drop_state, other_state = nnx.split(
    model, nnx.All(nnx.PathContains("lin"), nnx.Param), ...
)
nnx.state(model)

[[0.        2.8599195 0.        0.        0.       ]]


State({
  'b': VariableState( # 5 (20 B)
    type=Param,
    value=Array([0., 0., 0., 0., 0.], dtype=float32)
  ),
  'dropout': {
    'rngs': {
      'default': {
        'count': VariableState( # 1 (4 B)
          type=RngCount,
          value=Array(2, dtype=uint32),
          tag='default'
        ),
        'key': VariableState( # 1 (8 B)
          type=RngKey,
          value=Array((), dtype=key<fry>) overlaying:
          [0 2],
          tag='default'
        )
      },
      'dropout': {
        'count': VariableState( # 1 (4 B)
          type=RngCount,
          value=Array(1, dtype=uint32),
          tag='dropout'
        ),
        'key': VariableState( # 1 (8 B)
          type=RngKey,
          value=Array((), dtype=key<fry>) overlaying:
          [ 0 42],
          tag='dropout'
        )
      }
    }
  },
  'lin': {
    'b': VariableState( # 5 (20 B)
      type=Param,
      value=Array([0., 0., 0., 0., 0.], dtype=float32)
    ),
    'w': VariableState( # 10 (40 B)
      

In [2]:
drop_state

State({
  'lin': {
    'b': VariableState( # 5 (20 B)
      type=Param,
      value=Array([0., 0., 0., 0., 0.], dtype=float32)
    ),
    'w': VariableState( # 10 (40 B)
      type=Param,
      value=Array([[0.99568295, 0.5087098 , 0.01863182, 0.89239514, 0.29435277],
             [0.9759338 , 0.6530578 , 0.37484288, 0.9107307 , 0.20158172]],      dtype=float32)
    )
  }
})

In [3]:
import jax
import jax.numpy as jnp


class MyClass:
    def __init__(self, scale):
        self.scale = jnp.asarray(scale, dtype=jnp.float32)  # Differentiable attribute

    # Define a wrapper function that takes the instance as an argument
    # This function contains the primal computation logic
    def _scale_and_add_wrapped(instance, x, offset):
        return instance.scale * x + offset

    # Define the forward pass function for the custom VJP
    # This function takes the same arguments as the wrapped function, including the instance
    def _scale_and_add_fwd(instance, x, offset):
        # Compute the primal output
        primal_out = instance.scale * x + offset
        # Return primal output and residuals needed for the backward pass
        return primal_out, (instance.scale, x)  # Residuals: scale and x

    # Define the backward pass function for the custom VJP
    # This function takes the instance (due to nondiff_argnums), residuals, and output tangent
    def _scale_and_add_bwd(instance, residuals, g):
        # residuals contains (instance.scale, x) from the forward pass
        scale, x = residuals
        # g is the tangent with respect to the output of the primal function.
        # In the case of grad(sum(output)), g will be an array of ones with the shape of the output.

        # Compute tangents with respect to the differentiable inputs (x and offset).
        # Since 'instance' was marked as nondifferentiable (argnums=0), we don't
        # return a tangent for it. The tangents returned must correspond to the
        # differentiable arguments of the function wrapped by custom_vjp,
        # in the order they appear after the non-differentiable arguments.
        # The original wrapped function signature is _scale_and_add_wrapped(instance, x, offset).
        # With nondiff_argnums=(0,), we need to return tangents for x (arg 1) and offset (arg 2).

        # Derivative of (scale * x + offset) with respect to x is scale.
        # Since g is the gradient of a scalar (the sum) with respect to the output elements,
        # the gradient w.r.t x is element-wise scale * g.
        tangent_x = scale * g
        # Derivative of (scale * x + offset) with respect to offset is 1.0.
        # Similarly, the gradient w.r.t offset is element-wise 1.0 * g.
        tangent_offset = 1.0 * g

        # Return tangents corresponding to (x, offset)
        return (tangent_x, tangent_offset)

    # Apply jax.custom_vjp to the standalone wrapper function.
    # Mark the 'instance' argument (0th argument) as non-differentiable.
    _scale_and_add_custom_vjp = jax.custom_vjp(_scale_and_add_wrapped, nondiff_argnums=(0,))

    # Associate the forward and backward pass functions with the custom VJP definition.
    _scale_and_add_custom_vjp.defvjp(_scale_and_add_fwd, _scale_and_add_bwd)

    # Define the public member function that users will call.
    # This function simply calls the custom VJP-wrapped standalone function,
    # passing 'self' explicitly as the instance argument.
    def scale_and_add(self, x, offset):
        return self._scale_and_add_custom_vjp(self, x, offset)


# Example Usage
key = jax.random.PRNGKey(0)
instance = MyClass(scale=2.0)

x_val = jnp.array([1.0, 2.0, 3.0])
offset_val = jnp.array([0.5, 1.0, 1.5])

# Compute the value using the member function
# This calls the scale_and_add method, which in turn calls the custom_vjp wrapped function
output = instance.scale_and_add(x_val, offset_val)
print("Output:", output)


# Compute the gradient using jax.grad
# JAX's grad function is designed for scalar-output functions.
# Our scale_and_add function returns a vector (shape (3,)).
# To use grad, we need to wrap the call in a function that returns a scalar,
# for example, the sum of the output.
def scalar_output_fn(x, offset):
    return jnp.sum(instance.scale_and_add(x, offset))


# Now apply grad to this scalar-output function.
# The argnums here refer to the arguments of scalar_output_fn, which are x (0) and offset (1).
grad_fn = jax.grad(scalar_output_fn, argnums=(0, 1))

# Call the gradient function with the values of x and offset
gradients = grad_fn(x_val, offset_val)

print("Gradients w.r.t x and offset (of the sum of the output):", gradients)

# Verify gradients
# The function being differentiated is f(x, offset) = sum(scale * x + offset)
# f(x, offset) = sum(scale * x_i + offset_i) over i
# The derivative of f with respect to x_j is d/dx_j (sum(scale * x_i + offset_i))
# This sum is scale * x_0 + offset_0 + scale * x_1 + offset_1 + scale * x_2 + offset_2
# d/dx_j (scale * x_j) = scale. d/dx_j (other terms) = 0.
# So, the gradient with respect to each element of x is 'scale'.
expected_grad_x = instance.scale * jnp.ones_like(x_val)

# The derivative of f with respect to offset_j is d/d_offset_j (sum(scale * x_i + offset_i))
# d/d_offset_j (offset_j) = 1. d/d_offset_j (other terms) = 0.
# So, the gradient with respect to each element of offset is 1.0.
expected_grad_offset = 1.0 * jnp.ones_like(offset_val)

print("Expected gradient w.r.t x:", expected_grad_x)
print("Expected gradient w.r.t offset:", expected_grad_offset)

assert jnp.allclose(gradients[0], expected_grad_x)
assert jnp.allclose(gradients[1], expected_grad_offset)

print("\nGradients match expected values.")

Output: [2.5 5.  7.5]
Gradients w.r.t x and offset (of the sum of the output): (Array([2., 2., 2.], dtype=float32), Array([1., 1., 1.], dtype=float32))
Expected gradient w.r.t x: [2. 2. 2.]
Expected gradient w.r.t offset: [1. 1. 1.]

Gradients match expected values.


In [4]:
rngs = nnx.Rngs(params=0, dropout=1)


@nnx.split_rngs(splits=5, only="dropout")
def f(rngs: nnx.Rngs):
    print("Inside:")
    # rngs.dropout() # ValueError: fold_in accepts a single key...
    nnx.display(rngs)


f(rngs)

print("Outside:")
rngs.dropout()  # works!
nnx.display(rngs)

Inside:


Outside:


### Test random policy

In [5]:
from hcnn.policy.policy_mlp import PolicyMLP

rngs = nnx.Rngs(42, dropout=1)
policy = PolicyMLP.factory([2, 3, 2], 0.2, rngs=rngs)

random_state = nnx.state(policy, nnx.RngState)  # returns value, not reference
nnx.display(random_state)

nnx.display(policy)

In [6]:
print(policy(jnp.array([1, 2])))
print(policy(jnp.array([1, 2])))

[0.21044265 0.36397767]
[-0.01291138 -1.4362088 ]


In [7]:
nnx.display(policy)

In [8]:
nnx.display(random_state)

In [9]:
nnx.update(policy, random_state)
print(policy(jnp.array([1, 1])))
print(policy(jnp.array([1, 1])))

[0.26891333 0.24681942]
[ 0.08261265 -1.254725  ]


In [21]:
class MLP(nnx.Module):
    def __init__(self, din: int, dmid: int, dout: int, *, rngs: nnx.Rngs):
        self.linear1 = Linear(din, dmid, rngs=rngs)
        self.dropout = nnx.Dropout(rate=0.1, rngs=rngs)
        self.linear2 = Linear(dmid, dout, rngs=rngs)

    def __call__(self, x: jax.Array, rngs=None):
        x = nnx.gelu(self.dropout(self.linear1(x), rngs=rngs))
        return self.linear2(x)


model = MLP(2, 16, 5, rngs=nnx.Rngs(0, dropout=42))

batch_num = 20
x = jnp.ones((20, 2))


@nnx.vmap(in_axes=(None, 0, None), out_axes=0)
def forward(model, x: jnp.ndarray, rngs):
    return model(x, rngs=rngs)


y = forward(policy, x, rngs=nnx.clone(policy.rngs))
y

Array([[ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ]], dtype=float32)

In [30]:
tuple_list = [
    (jnp.array([1, 2, 3]), jnp.array([4, 5, 6])),
    (jnp.array([1, 2, 3]), jnp.array([4, 5, 6])),
    (jnp.array([1, 2, 3]), jnp.array([4, 5, 6])),
    (jnp.array([1, 2, 3]), jnp.array([4, 5, 6])),
]

lhs_list, rhs_list = zip(*tuple_list, strict=False)

all_lhs = jnp.concatenate(lhs_list)
all_rhs = jnp.concatenate(rhs_list)

all_lhs

Array([1, 2, 3, 1, 2, 3, 1, 2, 3, 1, 2, 3], dtype=int32)